# Heaviliate — Video Slideshow Produk (9:16)

Mengubah JSON bahan dari aplikasi Heaviliate menjadi video vertikal:
foto produk + caption besar + voiceover Bahasa Indonesia (edge-tts).

**Cara pakai:**
1. Jalankan sel 1 (pasang pustaka, ±1 menit, sekali saja).
2. Tempel JSON dari aplikasi ke sel 2, ganti isinya.
3. Runtime → Run all. Video jadi ±2–3 menit, otomatis terunduh.

Tanpa GPU, tanpa API key, tanpa biaya. Semua foto diambil dari URL yang
kamu tempel sendiri — tidak ada yang diunggah ke layanan lain.

In [ ]:
# Sel 1 — pasang pustaka
!pip -q install moviepy==1.0.3 edge-tts pillow requests
print("Pustaka siap.")

In [ ]:
# Sel 2 — BAHAN: tempel JSON dari aplikasi Heaviliate di sini, ganti isinya
BAHAN = r'''
{
  "versi": 1,
  "produk": { "nama": "Contoh Produk", "harga_tampil": "Rp 0", "url_affiliate": "" },
  "suara": "id-ID-ArdiNeural",
  "scene": [
    { "teks": "Ganti teks ini dari aplikasi.", "foto_url": "https://example.com/foto.jpg", "durasi_detik": 4 }
  ],
  "keluaran": { "lebar": 1080, "tinggi": 1920, "fps": 30, "nama_berkas": "slideshow-contoh.mp4" }
}
'''

In [ ]:
# Sel 3 — render
import asyncio, io, json, math, os, tempfile
import requests
import edge_tts
from PIL import Image, ImageDraw, ImageFont
from moviepy.editor import (AudioFileClip, CompositeVideoClip, ImageClip,
                            concatenate_videoclips)

bahan = json.loads(BAHAN)
W, H = bahan["keluaran"]["lebar"], bahan["keluaran"]["tinggi"]
FPS = bahan["keluaran"]["fps"]
NAMA = bahan["keluaran"]["nama_berkas"]
tmp = tempfile.mkdtemp()
print(f"{len(bahan['scene'])} scene | suara {bahan['suara']} | keluaran {NAMA}")

def unduh_foto(url):
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    # cover: isi penuh 1080x1920, potong tengah
    skala = max(W / img.width, H / img.height)
    img = img.resize((round(img.width * skala), round(img.height * skala)), Image.LANCZOS)
    kiri = (img.width - W) // 2
    atas = (img.height - H) // 2
    return img.crop((kiri, atas, kiri + W, atas + H))

def muat_font(ukuran):
    for jalur in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                  "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if os.path.exists(jalur):
            return ImageFont.truetype(jalur, ukuran)
    return ImageFont.load_default()

def bungkus_teks(teks, font, maks_lebar):
    baris, baris_ini = [], ""
    for kata in teks.split():
        coba = (baris_ini + " " + kata).strip()
        if font.getlength(coba) <= maks_lebar:
            baris_ini = coba
        else:
            if baris_ini: baris.append(baris_ini)
            baris_ini = kata
    if baris_ini: baris.append(baris_ini)
    return baris

def caption_png(teks, harga):
    """Caption band di bawah: teks putih di atas pita hitam tembus pandang."""
    font = muat_font(64)
    font_kecil = muat_font(44)
    maks = W - 120
    baris = bungkus_teks(teks, font, maks)
    tinggi_teks = len(baris) * 78
    tinggi_pita = tinggi_teks + (70 if harga else 30) + 60
    img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    d.rectangle([0, H - tinggi_pita, W, H], fill=(10, 8, 8, 210))
    y = H - tinggi_pita + 30
    for b in baris:
        d.text((60, y), b, font=font, fill=(255, 255, 255, 255))
        y += 78
    if harga:
        d.text((60, y + 8), harga, font=font_kecil, fill=(249, 115, 22, 255))
    jalur = os.path.join(tmp, "caption.png")
    img.save(jalur)
    return jalur

async def tts(teks, suara, jalur):
    await edge_tts.Communicate(teks, suara).save(jalur)

loop = asyncio.get_event_loop()
klip_scene = []
for i, sc in enumerate(bahan["scene"]):
    print(f"Scene {i+1}: {sc['teks'][:40]}...")
    foto = unduh_foto(sc["foto_url"])
    jalur_foto = os.path.join(tmp, f"foto{i}.jpg")
    foto.save(jalur_foto, quality=92)

    jalur_suara = os.path.join(tmp, f"suara{i}.mp3")
    loop.run_until_complete(tts(sc["teks"], bahan["suara"], jalur_suara))
    audio = AudioFileClip(jalur_suara)
    durasi = max(sc.get("durasi_detik", 4), audio.duration + 0.4)

    gambar = ImageClip(jalur_foto).set_duration(durasi)
    # Ken Burns: zoom pelan 1.00 -> 1.08
    gambar = gambar.resize(lambda t: 1 + 0.08 * (t / durasi))
    gambar = gambar.set_position(("center", "center"))
    cap = ImageClip(caption_png(sc["teks"], bahan["produk"].get("harga_tampil", "")))
    cap = cap.set_duration(durasi).set_position((0, "bottom"))
    klip = CompositeVideoClip([gambar, cap], size=(W, H)).set_duration(durasi)
    klip = klip.set_audio(audio.set_start(0.2))
    klip_scene.append(klip)

video = concatenate_videoclips(klip_scene, method="compose")
video = video.set_fps(FPS)
video.write_videofile(NAMA, codec="libx264", audio_codec="aac",
                      preset="fast", threads=2, logger=None)
print(f"SELESAI: {NAMA} ({os.path.getsize(NAMA)//1024} KB)")

In [ ]:
# Sel 4 — unduh hasil
from google.colab import files
files.download(NAMA)